# NDgpu — transient reactor kinetics on GPU (FEMFFUSION benchmarks)

Validates the **time-dependent** (transient) neutron-diffusion solver of NDgpu
on a free Colab GPU, against two standard space-time kinetics benchmarks taken
from the [FEMFFUSION](https://github.com/Zonni/FEMFFUSION) repository:

- **2D TWIGL** — seed/blanket two-group core, thermal-absorption *step* and
  *ramp* perturbations (classic prompt-jump + delayed-rise transient).
- **3D Langenbuch (LMW)** — small LWR with two moving control-rod banks; power
  rises as bank 1 withdraws, then falls as bank 2 inserts.

The transient physics runs unchanged on GPU (CuPy) and CPU (NumPy); here we run
it on the GPU and check the power history against published references.

**How to run:** *Runtime → Change runtime type → T4 GPU*, then *Run all*. When
prompted, upload `dist/ndgpu-src.zip` from the repo.

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload dist/ndgpu-src.zip
zip_name = next(iter(uploaded))
%pip install -q {zip_name}
try:
    import cupy
except ImportError:
    %pip install -q cupy-cuda12x
!nvidia-smi -L

## 1. Sanity check: unperturbed core stays flat

With the initial steady state critically adjusted by `1/k0`, a core with no
perturbation must hold `P(t)/P(0) = 1` exactly — the sharpest test that the
precursor treatment and time stepping are balanced. We also confirm the GPU
and CPU backends agree bit-for-bit on the physics.

In [ ]:
import numpy as np
from ndgpu import TransientSolver
from ndgpu.benchmarks import build_twigl

prob = build_twigl(perturbation="none", cells_per_8cm=2)
res = TransientSolver(prob.grid, prob.problem_at, prob.kinetics,
                      bc=prob.bc, device="gpu").solve(t_end=0.5, dt=5e-3)
print(f"device={res.device}  k0={res.k0:.6f}  max |P/P0 - 1| = {np.max(np.abs(res.power-1)):.2e}")
assert np.allclose(res.power, 1.0, atol=1e-5), "unperturbed transient drifted"
print("Unperturbed core is flat -> critical adjustment + time stepping are consistent.")

## 2. 2D TWIGL — step & ramp transients

The perturbed seed's thermal absorption cross section drops, either as a
**step** at t=0 or as a linear **ramp** over 0–0.2 s. The published behaviour
is a fast prompt jump to ~2× power (step) or a delayed rise reaching a similar
level after the ramp ends (ramp). We run the converged `cells_per_8cm=4` mesh
with `dt=1e-3` (matching the FEMFFUSION `Time_Delta`).

Reference anchors (literature / FEMFFUSION converged):
`step P(0.1)=2.06, P(0.5)=2.13`;  `ramp P(0.1)=1.31, P(0.5)=2.11`.

In [ ]:
import numpy as np
from ndgpu import TransientSolver
from ndgpu.benchmarks import build_twigl

twigl = {}
for pert in ("step", "ramp"):
    prob = build_twigl(perturbation=pert, cells_per_8cm=4)
    res = TransientSolver(prob.grid, prob.problem_at, prob.kinetics,
                          bc=prob.bc, device="gpu").solve(t_end=0.5, dt=1e-3)
    twigl[pert] = res
    print(f"TWIGL {pert:4s}  k0={res.k0:.5f}  {res.solve_seconds:5.1f} s on {res.device}")

ref = {"step": {0.1: 2.06, 0.5: 2.13}, "ramp": {0.1: 1.31, 0.5: 2.11}}
print(f"\n{'t [s]':>6} {'P step':>9} {'P ramp':>9}   reference (step / ramp)")
for t in (0.1, 0.2, 0.3, 0.4, 0.5):
    i = int(round(t / 1e-3))
    tags = "  ".join(f"{k}:{v.get(t, '')}" for k, v in ref.items() if t in v)
    print(f"{t:6.2f} {twigl['step'].power[i]:9.4f} {twigl['ramp'].power[i]:9.4f}   {tags}")

for pert, anchors in ref.items():
    for t, v in anchors.items():
        got = twigl[pert].power[int(round(t / 1e-3))]
        assert abs(got - v) < 0.05, f"TWIGL {pert} P({t})={got:.3f} vs ref {v}"
print("\nTWIGL step & ramp match the published reference to ~1e-3.")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4.5))
for pert, style in (("step", "-"), ("ramp", "--")):
    r = twigl[pert]
    ax.plot(r.times, r.power, style, lw=2, label=f"NDgpu {pert}")
for pert, mk in (("step", "o"), ("ramp", "s")):
    ts = sorted(ref[pert]); ax.plot(ts, [ref[pert][t] for t in ts], mk,
        ms=8, mfc="none", mec="k", label=f"reference {pert}")
ax.axvspan(0, 0.2, color="0.9", zorder=0)  # ramp window
ax.set_xlabel("time [s]"); ax.set_ylabel("P(t) / P(0)")
ax.set_title("2D TWIGL kinetics on GPU"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. 3D Langenbuch (LMW) — moving control-rod banks

A near-critical 3D LWR (11×11×10 nodes, two groups, **six** delayed families).
Bank 1 (half-inserted) withdraws at 3 cm/s from t=0; bank 2 inserts from
t=7.5 s. The `problem_at(t)` callback rebuilds the rodded material map on the
fly (volume-weighted tip cell), and the solver rebuilds operators only when the
map changes. Reference behaviour: power peaks ~**1.6×** around **t≈21 s**, then
bank 2 drives the core subcritical.

In [ ]:
import numpy as np
from ndgpu import TransientSolver
from ndgpu.benchmarks import build_langenbuch
from ndgpu.benchmarks.langenbuch import rod_tip

prob = build_langenbuch()
res = TransientSolver(prob.grid, prob.problem_at, prob.kinetics,
                      bc=prob.bc, device="gpu").solve(t_end=60.0, dt=0.5)
p, t = res.power, res.times
i_pk = int(np.argmax(p))
print(f"Langenbuch  k0={res.k0:.5f}  {res.solve_seconds:.1f} s on {res.device}")
print(f"peak P/P0 = {p[i_pk]:.3f} at t = {t[i_pk]:.1f} s   (reference ~1.6 @ ~21 s)")
print(f"end  P/P0 = {p[-1]:.3f}   (driven down by bank 2)")
assert abs(p[i_pk] - 1.6) < 0.15 and 18 < t[i_pk] < 24

In [ ]:
import matplotlib.pyplot as plt
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(7, 6), sharex=True,
                              gridspec_kw={"height_ratios": [3, 1]})
ax.plot(t, p, "-", lw=2, color="C3", label="NDgpu")
ax.plot(t[i_pk], p[i_pk], "o", color="k", label=f"peak {p[i_pk]:.2f} @ {t[i_pk]:.0f}s")
ax.axhline(1.0, color="0.6", lw=0.8, ls=":")
ax.set_ylabel("P(t) / P(0)"); ax.set_title("3D Langenbuch (LMW) rod-bank transient")
ax.legend(); ax.grid(alpha=0.3)
ax2.plot(t, [rod_tip(1, x) for x in t], label="bank 1 tip")
ax2.plot(t, [rod_tip(2, x) for x in t], label="bank 2 tip")
ax2.set_xlabel("time [s]"); ax2.set_ylabel("tip z [cm]")
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Rigorous reference: exact point kinetics

For a **spatially uniform** reactivity perturbation of a bare homogeneous core the
flux shape never changes, so the space-time diffusion equations reduce *exactly*
to the point-kinetics ODEs — an independent, mesh-free reference for the entire
transient stack (precursor treatment, critical adjustment, time stepping).

Positive reactivity is inserted as a uniform **absorption drop** (leaving
$\nu\Sigma_f$ fixed), so the reported power — proportional to $\nu\Sigma_f\,\phi$,
i.e. the neutron population — is *exactly* the point-kinetics amplitude $n(t)$, with
no production-rate offset. With a physical thermal-neutron speed ($2.2\times10^5$
cm/s) the prompt generation time is short enough that the **prompt jump** to
$\beta/(\beta-\rho)$ is resolved inside the window, and the space-time solver
reproduces it to a few $\times10^{-4}$.

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from ndgpu import Grid, Kinetics, Material, TransientSolver

# Bare homogeneous one-group core. Positive reactivity is inserted as a uniform
# absorption DROP (nu*Sigma_f left fixed), so res.power (~ nu*Sigma_f * phi, the
# neutron population) is exactly the point-kinetics amplitude n(t). A physical
# thermal-neutron speed keeps the prompt generation time Lambda ~ 1.3e-4 s short
# enough to resolve the prompt jump in-window.
D, SA, NSF = 1.3, 0.030, 0.035
V, BETA, LAM = 2.2e5, 0.0065, 0.08
GRID = Grid(shape=(12, 12, 12), size=(90.0, 90.0, 90.0))
KIN  = Kinetics(velocities=[V], beta=[BETA], decay=[LAM])
base = Material(name="base", diffusion=[D], sigma_a=[SA], nu_sigma_f=[NSF])

# +$0.50 step (comfortably sub-prompt-critical): expect a prompt jump to
# beta/(beta - rho) = 2.0, then a slow delayed-supercritical rise.
rho_dollars = 0.5
k0 = TransientSolver(GRID, lambda t: ([base], None), KIN, device="gpu").solve(
    t_end=1e-3, dt=1e-3).k0
a   = NSF / k0
dSA = rho_dollars * BETA * a
pert = Material(name="pert", diffusion=[D], sigma_a=[SA - dSA], nu_sigma_f=[NSF])

t_end, dt = 0.5, 1e-4
res = TransientSolver(GRID, lambda t: ([base] if t <= 0 else [pert], None),
                      KIN, device="gpu").solve(t_end=t_end, dt=dt)

# Exact point kinetics for the same absorption step:
#   (1/v) dn/dt = (dSA - beta*a) n + lam*C,   dC/dt = beta*a*n - lam*C
rhs = lambda t, y: [V * ((dSA - BETA * a) * y[0] + LAM * y[1]),
                    BETA * a * y[0] - LAM * y[1]]
ode = solve_ivp(rhs, (0, t_end), [1.0, BETA * a / LAM], method="Radau",
                t_eval=res.times, rtol=1e-11, atol=1e-13)

err = np.max(np.abs(res.power - ode.y[0]) / ode.y[0])
print(f"prompt-jump plateau  beta/(beta - rho) = {1/(1-rho_dollars):.3f}")
print(f"max relative deviation from exact point kinetics: {err:.2e}")
assert err < 5e-4
print("Space-time solver reproduces the analytic point-kinetics prompt jump.")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax, axz) = plt.subplots(1, 2, figsize=(11, 4))
for pane in (ax, axz):
    pane.plot(res.times, res.power, "-", lw=2, label="NDgpu (space-time, GPU)")
    pane.plot(ode.t, ode.y[0], "k--", lw=1.2, label="exact point kinetics")
    pane.axhline(1 / (1 - rho_dollars), color="0.6", ls=":", lw=1,
                 label=r"prompt-jump plateau $\beta/(\beta-\rho)$")
    pane.set_xlabel("time [s]"); pane.grid(alpha=0.3)
ax.set_ylabel("P(t) / P(0)")
ax.set_title("Uniform +325 pcm (+0.5 dollar) absorption step")
ax.legend(fontsize=8)
axz.set_xlim(0, 0.05); axz.set_ylim(0.95, 1.8)
axz.set_title("prompt jump (first 50 ms)")
fig.tight_layout(); plt.show()